In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('../data/raw/store-data.csv')

In [3]:
rows, columns = df.shape
print(f'[+] Raw Dataset Contain {rows} Rows And {columns} Columns')

[+] Raw Dataset Contain 10064 Rows And 21 Columns


In [4]:
def get_duplicated_statics():
    duplicatedRows = df.duplicated()
    print(f'[+] {duplicatedRows.sum()} => {(duplicatedRows.mean() * 100):.2f}% of rows are duplicated')
get_duplicated_statics()

[+] 34 => 0.34% of rows are duplicated


In [ ]:
ids = [id for id, boolean in enumerate(df.duplicated()) if boolean == True]
df.drop(ids, inplace=True)
df.reset_index(drop=True)
pd.to_datetime(df['Ship Date'], format='mixed').dt.strf

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3.0,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2.0,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10025,1281,CA-2016-160815,9/5/2016,9/6/2016,First Class,TR-21325,TOBY RITTER,Consumer,United States,Cedar Rapids,...,52402.0,Central,TEC-PH-10003505,Technology,Phones,Geemarc AmpliPOWER60,278.4000,3.0,0.00,80.7360
10026,7699,CA-2017-151799,12/14/2017,12/18/2017,Standard Class,BF-11170,BEN FERRER,Home Office,United States,Lawrence,...,1841.0,East,OFF-ST-10002790,Office Supplies,Storage,Safco Industrial Shelving,73.8500,1.0,0.00,2.2155
10027,8597,CA-2014-111934,5/5/2014,5/7/2014,First Class,GD-14590,GIULIETTA DORTCH,Corporate,United States,Arlington,...,22204.0,South,OFF-PA-10000474,Office Supplies,Paper,Easy-staple paper,35.4400,1.0,0.00,16.6568
10028,4962,CA-2014-156587,3/7/2014,3/8/2014,First Class,AB-10015,NaN,Consumer,United States,Seattle,...,98103.0,West,FUR-CH-10004477,Furniture,Chairs,"Global Push Button Manager's Chair, Indigo",48.7120,1.0,0.20,5.4801


In [6]:
get_duplicated_statics()

[+] 0 => 0.00% of rows are duplicated


In [7]:
for column in df.columns:
    nullValues = df[column].isnull()
    print(f'[+] Column "{column}" Contain: {nullValues.sum()} Null Values => {(nullValues.mean() * 100):.2f}%')

[+] Column "Row ID" Contain: 0 Null Values => 0.00%
[+] Column "Order ID" Contain: 0 Null Values => 0.00%
[+] Column "Order Date" Contain: 0 Null Values => 0.00%
[+] Column "Ship Date" Contain: 101 Null Values => 1.01%
[+] Column "Ship Mode" Contain: 301 Null Values => 3.00%
[+] Column "Customer ID" Contain: 0 Null Values => 0.00%
[+] Column "Customer Name" Contain: 317 Null Values => 3.16%
[+] Column "Segment" Contain: 0 Null Values => 0.00%
[+] Column "Country" Contain: 0 Null Values => 0.00%
[+] Column "City" Contain: 0 Null Values => 0.00%
[+] Column "State" Contain: 0 Null Values => 0.00%
[+] Column "Postal Code" Contain: 199 Null Values => 1.98%
[+] Column "Region" Contain: 0 Null Values => 0.00%
[+] Column "Product ID" Contain: 0 Null Values => 0.00%
[+] Column "Category" Contain: 0 Null Values => 0.00%
[+] Column "Sub-Category" Contain: 0 Null Values => 0.00%
[+] Column "Product Name" Contain: 0 Null Values => 0.00%
[+] Column "Sales" Contain: 200 Null Values => 1.99%
[+] Colum

In [9]:
# I used at with index (0 -> len - 1) to get the exact value of a column like df.at[54, 'Row ID']
df.sort_values('Order Date', inplace=True)
null_ship_date_row_ids = [np.int16(df.at[id, 'Row ID']) for id, value in enumerate(df['Ship Date'].isnull()) if value == True]

In [29]:
null_ship_date_row_ids = [np.int16(df.iloc[id]['Row ID']) for id, value in enumerate(df['Ship Date'].isnull()) if value == True]

In [ ]:
def get_most_close_order_date(row_id, mean_duration_between_order_shap_dates):
    past_dates = np.full(3, mean_duration_between_order_shap_dates)
    past_dates_values = 0
    past_dates_id = row_id
    future_dates = np.full(3, mean_duration_between_order_shap_dates)
    future_dates_values = 0
    future_dates_id = row_id
    while past_dates_values < 4 and past_dates_id >= 0:
        past_dates_id -= 1
        ship_and_order_dates = df[['Order Date', 'Ship Date']][df['Row ID'] == past_dates_id]
        if ship_and_order_dates['Ship Date'].isnull(): continue
        past_dates[past_dates_values] = pd.to_datetime(ship_and_order_dates['Shape Date'].iloc[0]).timestamp() - pd.to_datetime(ship_and_order_dates['Order Date'].iloc[0]).timestamp()

    while future_dates_values < 4 and future_dates_id >= 0:
        future_dates_id -= 1
        ship_and_order_dates = df[['Order Date', 'Ship Date']][df['Row ID'] == future_dates_id]
        if ship_and_order_dates['Ship Date'].isnull(): continue
        future_dates[future_dates_values] = pd.to_datetime(ship_and_order_dates['Shape Date'].iloc[0]).timestamp() - pd.to_datetime(ship_and_order_dates['Order Date'].iloc[0]).timestamp()


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
5067,5068,CA-2017-160934,1/15/2017,NaN,Standard Class,TT-21460,Tonja Turnell,Home Office,United States,Los Angeles,...,90008.0,West,OFF-AR-10003504,Office Supplies,Art,Newell 347,21.4,5.0,0.0,6.206


In [62]:
df[['Ship Date', 'Order Date', 'Row ID']].min()

Ship Date     1/1/2010
Order Date    1/1/2017
Row ID               1
dtype: object

In [64]:
df[df['Ship Date'] < df['Order Date']]

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
4939,4940,CA-2014-157147,1/13/2014,1/1/2010,Standard Class,BD-11605,Brian Dahlen,Consumer,United States,San Francisco,...,94109.0,West,OFF-AR-10003514,Office Supplies,Art,4009 Highlighters by Sanford,19.900,5.0,0.0,6.5670
5348,5349,CA-2015-149811,1/4/2015,1/10/2015,Standard Class,CS-12250,Chris Selesnick,Corporate,United States,Woodbury,...,55125.0,Central,OFF-BI-10003676,Office Supplies,Binders,"GBC Standard Recycled Report Covers, Clear Pla...",32.340,3.0,0.0,15.5232
5349,5350,CA-2015-149811,1/4/2015,1/10/2015,Standard Class,CS-12250,Chris Selesnick,Corporate,United States,Woodbury,...,55125.0,Central,OFF-PA-10004082,Office Supplies,Paper,Adams Telephone Message Book w/Frequently-Call...,39.900,5.0,0.0,19.9500
1759,1760,CA-2014-141817,1/5/2014,1/12/2014,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,...,19143.0,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536,3.0,0.2,4.8840
8375,8376,CA-2015-158701,1/5/2015,1/10/2015,Standard Class,JL-15175,James Lanier,Home Office,United States,San Francisco,...,94110.0,West,OFF-BI-10001294,Office Supplies,Binders,Fellowes Binding Cases,56.160,6.0,0.2,17.5500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7283,7284,CA-2017-169439,9/9/2017,9/13/2017,Standard Class,LC-17140,Logan Currie,Consumer,United States,Cleveland,...,44105.0,East,OFF-AR-10001374,Office Supplies,Art,"BIC Brite Liner Highlighters, Chisel Tip",25.920,5.0,0.2,3.8880
6255,6256,CA-2017-139444,9/9/2017,9/15/2017,Standard Class,GK-14620,Grace Kelly,Corporate,United States,Plano,...,75023.0,Central,OFF-LA-10000134,Office Supplies,Labels,Avery 511,9.856,4.0,0.2,3.4496
2399,2400,CA-2017-147361,9/9/2017,9/15/2017,Standard Class,SB-20290,Sean Braxton,Corporate,United States,Florence,...,29501.0,South,OFF-ST-10001809,Office Supplies,Storage,Fellowes Officeware Wire Shelving,628.810,7.0,0.0,12.5762
5441,5442,CA-2017-155929,9/9/2017,9/13/2017,Standard Class,AI-10855,Arianne Irving,Consumer,United States,Glendale,...,85301.0,West,OFF-PA-10000859,Office Supplies,Paper,Unpadded Memo Slips,6.368,2.0,0.2,2.3880


In [37]:
tst = df[['Order Date', 'Ship Date']][df['Row ID'] == 50]

In [51]:
pd.to_datetime(tst['Order Date'].iloc[0]).timestamp()

1429315200.0